# Titanic 모델 개선: Feature 추가 가치, 비선형 모델, Threshold와 Kaggle 제출

**학습일:** 2026-07-22  
**목표:** 기존 baseline feature를 유지할지 검토하고, 비선형 모델과 비교한 뒤 Kaggle 제출 결과까지 확인한다.

## 오늘의 실험 원칙

- 모든 실험에서 동일한 5-fold Stratified CV 분할을 사용한다.
- feature 비교 단계에서는 Logistic Regression과 threshold `0.5`를 고정한다.
- 모델 비교 단계에서는 동일한 baseline feature와 threshold `0.5`를 사용한다.
- 최종 모델 후보가 정해진 뒤에만 threshold를 다시 비교한다.

이렇게 feature, 모델, threshold를 한꺼번에 바꾸지 않고 각각의 효과를 구분한다.


## 1. 라이브러리와 데이터 불러오기


In [49]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("Ready")


Ready


In [4]:
train = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/titanic/test.csv")
gender_submission = pd.read_csv(
    "/kaggle/input/competitions/titanic/gender_submission.csv"
)

print(train.columns)

train.isna().mean().mul(100).sort_values(ascending=False)


Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
Cabin          77.104377
Embarked        0.224467
dtype: float64

## 2. Feature 추가 가치 비교

아직 사용하지 않은 변수는 `Name`, `Ticket`, `Fare`, `Embarked`였다.

- `Name`과 `Ticket` 원문은 고유값이 많아 이번 실험에서는 제외한다.
- `Fare`는 `Pclass`와 정보가 겹칠 수 있다는 가설을 확인한다.
- `Embarked`가 기존 feature에 추가 정보를 제공하는지 확인한다.
- `Name`에서 `Title`을 추출하는 실험은 이후 후보로 남겨 둔다.


In [11]:
# Fare와 Embarked의 기본 분포 확인
print(train["Fare"].describe())
print(train["Embarked"].describe())

train["Embarked"].value_counts()


count    891.000000
mean      32.204208
std       49.693429
min        0.000000
25%        7.910400
50%       14.454200
75%       31.000000
max      512.329200
Name: Fare, dtype: float64
count     889
unique      3
top         S
freq      644
Name: Embarked, dtype: object


Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [ ]:
# Baseline: Sex, Pclass, FamilySize, AgeGroup
def create_features(data):
    result = data.copy()

    # 본인을 포함한 전체 가족 규모
    result["FamilySize"] = result["SibSp"] + result["Parch"] + 1

    # Age 원값 대신 사용할 연령 구간
    result["AgeGroup"] = pd.cut(
        result["Age"],
        bins=[0, 6, 20, 40, 60, np.inf],
        labels=["Preschool", "Teen", "20-39", "40-59", "60+"],
        right=False,
    )
    return result


train_fe = create_features(train)
test_fe = create_features(test)

baseline_features = ["Sex", "Pclass", "FamilySize", "AgeGroup"]

X_baseline = train_fe[baseline_features]
X_embarked = train_fe[baseline_features + ["Embarked"]]
X_fare = train_fe[baseline_features + ["Fare"]]
X_embarked_fare = train_fe[baseline_features + ["Embarked", "Fare"]]

y = train_fe["Survived"]


In [ ]:
# 전처리 Pipeline
numerical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)


def build_model_pipeline(num_features, cat_features, estimator):
    preprocessor = ColumnTransformer(
        [
            ("num", numerical_transformer, num_features),
            ("cat", categorical_transformer, cat_features),
        ]
    )

    return Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )


In [ ]:
# 모든 실험에서 동일한 CV 분할을 사용한다.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

logistic_model = LogisticRegression(max_iter=1000, random_state=42)

baseline_pipeline = build_model_pipeline(
    ["FamilySize"],
    ["Sex", "Pclass", "AgeGroup"],
    clone(logistic_model),
)
embarked_pipeline = build_model_pipeline(
    ["FamilySize"],
    ["Sex", "Pclass", "AgeGroup", "Embarked"],
    clone(logistic_model),
)
fare_pipeline = build_model_pipeline(
    ["FamilySize", "Fare"],
    ["Sex", "Pclass", "AgeGroup"],
    clone(logistic_model),
)
embarked_fare_pipeline = build_model_pipeline(
    ["FamilySize", "Fare"],
    ["Sex", "Pclass", "AgeGroup", "Embarked"],
    clone(logistic_model),
)


def get_oof_prediction(X, pipeline):
    return cross_val_predict(
        estimator=pipeline,
        X=X,
        y=y,
        cv=cv,
        method="predict",
        n_jobs=-1,
    )


oof_pred_baseline = get_oof_prediction(X_baseline, baseline_pipeline)
oof_pred_embarked = get_oof_prediction(X_embarked, embarked_pipeline)
oof_pred_fare = get_oof_prediction(X_fare, fare_pipeline)
oof_pred_embarked_fare = get_oof_prediction(
    X_embarked_fare,
    embarked_fare_pipeline,
)


In [54]:
def result_df(model_name, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()

    return pd.DataFrame(
        [
            {
                "model": model_name,
                "TN": tn,
                "FP": fp,
                "FN": fn,
                "TP": tp,
                "accuracy": accuracy_score(y, pred),
                "recall": recall_score(y, pred),
                "precision": precision_score(y, pred),
                "f1": f1_score(y, pred),
                "predicted_survivors": pred.sum(),
            }
        ]
    )


result_baseline = result_df("baseline", oof_pred_baseline)
result_embarked = result_df("+Embarked", oof_pred_embarked)
result_fare = result_df("+Fare", oof_pred_fare)
result_embarked_fare = result_df(
    "+Embarked, +Fare",
    oof_pred_embarked_fare,
)

feature_comparison = pd.concat(
    [
        result_baseline,
        result_embarked,
        result_fare,
        result_embarked_fare,
    ],
    ignore_index=True,
)

feature_comparison


,model,TN,FP,FN,TP,accuracy,recall,precision,f1,predicted_survivors
0,baseline,482,67,100,242,0.812570,0.707602,0.783172,0.743472,309
1,+Embarked,481,68,98,244,0.813692,0.713450,0.782051,0.746177,312
2,+Fare,489,60,107,235,0.812570,0.687135,0.796610,0.737834,295
3,"+Embarked, +Fare",481,68,100,242,0.811448,0.707602,0.780645,0.742331,310


### Feature 비교 해석

- `Embarked`를 추가하면 F1이 `0.7435 → 0.7462`로 아주 조금 상승했지만 승객 1~3명 수준의 차이다.
- `Fare`를 추가하면 FP는 줄었지만 FN이 늘어 recall과 F1이 하락했다.
- `Embarked + Fare`도 baseline보다 개선되지 않았다.

따라서 추가 feature의 뚜렷한 가치가 확인되지 않았고, 단순성을 고려해 기존 baseline을 유지한다.

> **확정 feature:** `Sex`, `FamilySize`, `Pclass`, `AgeGroup`


## 3. Logistic Regression과 Gradient Boosting 비교

이전 실험에서 Decision Tree, Random Forest, Extra Trees는 이미 비교했다. 이번에는 동일한 baseline feature로 Gradient Boosting을 추가 비교한다.

Random Forest가 여러 트리를 비교적 독립적으로 학습해 결과를 종합한다면, Gradient Boosting은 작은 트리를 순차적으로 추가하면서 앞선 모델의 오류를 보완한다. 이를 통해 Logistic Regression이 직접 표현하지 못한 비선형 관계나 feature 간 상호작용을 학습할 수 있다.


In [61]:
gradient_boosting = GradientBoostingClassifier(random_state=42)

gb_pipeline = build_model_pipeline(
    ["FamilySize"],
    ["Sex", "Pclass", "AgeGroup"],
    gradient_boosting,
)

gb_oof_pred = cross_val_predict(
    estimator=gb_pipeline,
    X=X_baseline,
    y=y,
    cv=cv,
    method="predict",
    n_jobs=-1,
)

gb_result = result_df("Gradient Boosting", gb_oof_pred)

model_comparison = pd.concat(
    [result_baseline, gb_result],
    ignore_index=True,
)

model_comparison


,model,TN,FP,FN,TP,accuracy,recall,precision,f1,predicted_survivors
0,baseline,482,67,100,242,0.812570,0.707602,0.783172,0.743472,309
1,Gradient Boosting,491,58,101,241,0.821549,0.704678,0.806020,0.751950,299


### 모델 비교 해석

- accuracy: `0.8126 → 0.8215`
- precision: `0.7832 → 0.8060`
- F1: `0.7435 → 0.7520`
- recall: `0.7076 → 0.7047`로 거의 동일
- 전체 오분류: `167 → 159`, 8명 감소

Gradient Boosting은 생존자 recall을 거의 유지하면서 FP를 9명 줄였다. OOF 결과만 보면 Gradient Boosting이 더 좋은 최종 모델 후보다.


## 4. Gradient Boosting Threshold 비교


In [64]:
gb_oof_proba = cross_val_predict(
    estimator=gb_pipeline,
    X=X_baseline,
    y=y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

thresholds = [0.55, 0.50, 0.475, 0.45, 0.40, 0.35, 0.30]
threshold_results = []

for threshold in thresholds:
    threshold_pred = (gb_oof_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, threshold_pred).ravel()

    threshold_results.append(
        {
            "threshold": threshold,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "accuracy": accuracy_score(y, threshold_pred),
            "recall": recall_score(y, threshold_pred),
            "precision": precision_score(
                y,
                threshold_pred,
                zero_division=0,
            ),
            "f1": f1_score(y, threshold_pred),
            "predicted_survivors": threshold_pred.sum(),
        }
    )

threshold_summary = pd.DataFrame(threshold_results)
threshold_summary


,threshold,TN,FP,FN,TP,accuracy,recall,precision,f1,predicted_survivors
0,0.550,499,50,111,231,0.819304,0.675439,0.822064,0.741573,281
1,0.500,491,58,101,241,0.821549,0.704678,0.806020,0.751950,299
2,0.475,481,68,96,246,0.815937,0.719298,0.783439,0.750000,314
3,0.450,480,69,95,247,0.815937,0.722222,0.781646,0.750760,316
4,0.400,459,90,85,257,0.803591,0.751462,0.740634,0.746009,347
5,0.350,435,114,68,274,0.795735,0.801170,0.706186,0.750685,388
6,0.300,421,128,63,279,0.785634,0.815789,0.685504,0.744993,407


### Threshold 비교 해석

Gradient Boosting에서는 threshold `0.5`가 비교한 값 중 accuracy, precision, F1이 가장 높았다.

- `0.50 → 0.45`: recall은 `0.7047 → 0.7222`로 상승한다.
- 그러나 accuracy, precision, F1은 모두 소폭 하락한다.
- 특별히 생존자 recall을 높여야 한다는 실제 비용 기준이 없으므로 기본 threshold `0.5`를 유지한다.

이전 Logistic Regression에서 recall과 F1을 위해 선택했던 `0.45`가 Gradient Boosting에서도 최선인 것은 아니었다. **Threshold는 모델과 평가 목적에 따라 다시 판단해야 한다.**


## 5. 전체 train 학습과 Kaggle 제출 파일 생성

`cross_val_predict()`는 각 fold의 OOF 예측을 만들기 위한 함수이므로, Kaggle 제출 전에는 선택한 pipeline을 전체 train 데이터로 다시 학습한다.

비교를 위해 다음 세 파일을 생성한다.

1. Gradient Boosting, threshold `0.5`
2. Logistic Regression, threshold `0.5`
3. Logistic Regression, threshold `0.45`


In [ ]:
final_features = ["Sex", "FamilySize", "Pclass", "AgeGroup"]

X_final = train_fe[final_features]
y_final = train_fe["Survived"]
X_test_final = test_fe[final_features]


def make_submission(pipeline, threshold, file_name):
    final_pipeline = clone(pipeline)
    final_pipeline.fit(X_final, y_final)

    test_proba = final_pipeline.predict_proba(X_test_final)[:, 1]
    test_prediction = (test_proba >= threshold).astype(int)

    submission = pd.DataFrame(
        {
            "PassengerId": test["PassengerId"],
            "Survived": test_prediction,
        }
    )

    output_path = Path("/kaggle/working") / file_name
    submission.to_csv(output_path, index=False)

    print(file_name)
    print("shape:", submission.shape)
    print("missing values:\n", submission.isna().sum())
    print("prediction counts:\n", submission["Survived"].value_counts())
    print("saved to:", output_path)
    print()

    return submission, output_path


gb_submission, gb_output_path = make_submission(
    gb_pipeline,
    threshold=0.5,
    file_name="submission_gradient_boosting_050.csv",
)

lr_submission_050, lr_output_path_050 = make_submission(
    baseline_pipeline,
    threshold=0.5,
    file_name="submission_logistic_regression_050.csv",
)

lr_submission_045, lr_output_path_045 = make_submission(
    baseline_pipeline,
    threshold=0.45,
    file_name="submission_logistic_regression_045.csv",
)


## 6. Kaggle 제출 결과

| 모델 | Threshold | OOF Accuracy | Kaggle Score |
|---|---:|---:|---:|
| Gradient Boosting | 0.50 | **0.82155** | 0.76794 |
| Logistic Regression | 0.50 | 0.81257 | **0.77272** |
| Logistic Regression | 0.45 | 0.80696 | 0.75598 |

### 해석

1. **복잡한 모델이 항상 새로운 데이터에서 더 좋은 것은 아니다.**  
   Gradient Boosting은 OOF accuracy에서 Logistic Regression보다 높았지만 Kaggle test에서는 소폭 낮았다. 두 Kaggle 점수의 차이는 약 2명의 예측에 해당하므로 결정적인 차이라고 보기는 어렵다.

2. **CV 결과와 Kaggle test 결과는 다를 수 있다.**  
   데이터가 작아 fold와 test 구성에 따라 성능이 흔들릴 수 있고, 반복적인 실험 과정에서 validation 결과에 대한 선택 편향도 생길 수 있다.

3. **좋은 threshold는 평가 목적에 따라 달라진다.**  
   Logistic Regression의 threshold `0.45`는 OOF recall과 F1을 높였지만 accuracy는 낮췄다. Accuracy로 평가하는 Kaggle에서도 `0.5`보다 점수가 낮았다.

### 최종 제출 선택

```text
Features: Sex, FamilySize, Pclass, AgeGroup
Model: Logistic Regression
Threshold: 0.5
Kaggle score: 0.77272
```

OOF 결과만으로는 Gradient Boosting이 최종 후보였지만, 실제 Kaggle 제출 비교에서는 단순한 Logistic Regression이 소폭 높았다.


## 7. 오늘의 결론과 다음 단계

### 오늘 확인한 내용

- `Fare`와 `Embarked`는 기존 baseline에 뚜렷한 추가 가치를 제공하지 않았다.
- Gradient Boosting은 OOF accuracy·precision·F1을 개선했지만 Kaggle 점수는 Logistic Regression보다 낮았다.
- Logistic Regression threshold `0.45`는 recall을 높이는 선택이지만, Kaggle accuracy에는 불리했다.
- 최종 제출은 Logistic Regression과 threshold `0.5`로 결정했다.

### 다음에 해볼 일

1. `Name` 원문 대신 `Mr`, `Mrs`, `Miss`, `Master` 등의 `Title`을 추출해 추가 가치를 한 번만 비교한다.
2. 개선이 뚜렷하지 않으면 Titanic 프로젝트를 마무리한다.
3. 신용위험 프로젝트에서 Logistic Regression과 LightGBM/XGBoost 같은 비선형 모델을 비교한다.

Titanic처럼 작은 데이터에서 leaderboard 점수를 보며 feature, threshold, hyperparameter를 계속 바꾸면 과적합될 수 있다. 따라서 `Title` 실험 이후에는 결과가 크지 않다면 추가적인 leaderboard tuning보다 프로젝트의 실험 과정과 해석을 정리하는 데 집중한다.
